## Hyperparameter optimization

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import ParameterGrid
from utils import extract, Close_price, log_return, feature_engineering_rf, WFCV

TEST_TICKERS = [
    'GC1 Comdty',
    'BLQ2TRUU Index',
    'GBPJPY Curncy',
    'M4EUHDVD Index',
    'HFRXM Index',
    'GB1M Index',
    'QW2I Index',
    'FNER Index',
    'SPX Index',
]

def stats_forecasting_svm(df_all, name_ticker, model_params, step_size=50, fold_size=200):
    """Calcule les stats et la P-Value de la pente pour un ticker."""
    df_ticker = extract(df_all, name_ticker)
    df_ticker['Close'] = Close_price(df_ticker)
    df_ticker['log_return'] = log_return(df_ticker)
    df_ticker = feature_engineering_rf(df_ticker)

    X = df_ticker.drop(columns=['return', 'log_return', 'Close'])
    y = df_ticker['log_return']

    model = make_pipeline(StandardScaler(), SVR(**model_params))
    
    y_pred, y_truth, mse_tab, r2 = WFCV(X, y, model, step_size=step_size, fold_size=fold_size)
    
    reg = sm.OLS(y_truth, sm.add_constant(y_pred)).fit()

    return {
        'MSE': float(np.mean(mse_tab)) if len(mse_tab) else np.nan,
        'OLS_R2': float(reg.rsquared),
        'P_Value_Slope': float(reg.pvalues[1]) # p-value de l'indice 1 correspond à la pente[cite: 1, 2]
    }

def search_svm_refined(df_all, tickers, param_grid, step_size=50, fold_size=200):
    """Grid Search incluant la significativité statistique."""
    search_rows = []
    configs = list(ParameterGrid(param_grid))
    
    for i, params in enumerate(configs, start=1):
        res_list = [stats_forecasting_svm(df_all, t, params, step_size, fold_size) for t in tickers]
        
        mean_mse = np.mean([r['MSE'] for r in res_list])
        mean_r2 = np.mean([r['OLS_R2'] for r in res_list])
        mean_p_val = np.mean([r['P_Value_Slope'] for r in res_list])
        
        search_rows.append({
            'config_id': i,
            **params,
            'mean_mse': mean_mse,
            'mean_ols_r2': mean_r2,
            'mean_p_value_slope': mean_p_val
        })
        print(f"Config {i}: R2={mean_r2:.4f}, P-Val={mean_p_val:.4f}")

    return pd.DataFrame(search_rows)


df_results_svm = search_svm_refined(df, TEST_TICKERS, param_grid_svm, step_size=50, fold_size=200)

# --- LOGIQUE DE SÉLECTION ---

# 1. On filtre les configurations statistiquement significatives (p < 0.05)
# Cela garantit que le R2 n'est pas dû au hasard[cite: 1]
significant_configs = df_results_svm[df_results_svm['mean_p_value_slope'] < 0.05]

if not significant_configs.empty:
    # 2. Parmi les significatives, on prend le meilleur R2
    best_config = significant_configs.sort_values('mean_ols_r2', ascending=False).iloc[0]
    print("\nSÉLECTION : Meilleur R2 significatif (P-Value < 0.05)")
else:
    # 3. Si aucune n'est significative, on se rabat sur la MSE la plus basse (Prudence)
    best_config = df_results_svm.sort_values('mean_mse').iloc[0]
    print("\nSÉLECTION : Aucune config significative, choix par MSE minimale")

print(best_config)

c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Config 1: R2=0.0210, P-Val=0.0507


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Config 2: R2=0.0006, P-Val=0.5290


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


...

Config 4: R2=0.0010, P-Val=0.5274


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Config 5: R2=0.0200, P-Val=0.0507


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Config 6: R2=0.0014, P-Val=0.5274

SÉLECTION : Aucune config significative, choix par MSE minimale
config_id                    1
C                          0.1
epsilon                   0.01
gamma                    scale
kernel                     rbf
mean_mse              0.000285
mean_ols_r2           0.020953
mean_p_value_slope    0.050741
Name: 0, dtype: object


## Training on all dataset

In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from tqdm import tqdm
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import os
from utils import extract, Close_price, log_return, feature_engineering_rf, WFCV

BEST_PARAMS_SVM = {
    'C': 0.1,
    'epsilon': 0.01,
    'gamma': 'scale',
    'kernel': 'rbf'
}

COLS_TO_DROP = [
    'KOCRD Index', 'NOBRON Index', 'CABROVER Index', 'EB03/3M Index', 'BZSTSETA Index',
    'MXNZ0EN Index', 'EGDIMW Index', 'UX1 Index', 'VXEEM Index', 'VXEFA Index', 'VXEWZ Index',
    'VXFXI Index', 'VXGDX Index', 'VXSLV Index', 'W 1 Comdty', 'W1DOW Index', 'WBI Index',
    'FNRETTM Index', 'FNREU Index', 'WIG Index', 'WIGCON Index', 'WIGFOOD Index', 'XU100 Index',
    'XUHIZ Index', 'XUMAL Index', 'XUSIN Index', 'XUTEK Index', 'YW1 Comdty', 'ZAREUR Curncy',
    'ZARJPY Curncy', 'ZARUSD Curncy'
]

def stats_forecasting_svm_custom(df_all, name_ticker, model_params, step_size=50, fold_size=200):
    """
    Exécute le pipeline SVM complet pour un ticker donné avec Walk-Forward CV.
    """
    df_ticker = extract(df_all, name_ticker)
    df_ticker['Close'] = Close_price(df_ticker)
    df_ticker['log_return'] = log_return(df_ticker)
    df_ticker = feature_engineering_rf(df_ticker)

    X = df_ticker.drop(columns=['return', 'log_return', 'Close'])
    y = df_ticker['log_return']

    model = make_pipeline(StandardScaler(), SVR(**model_params))
    
    y_pred, y_truth, mse_tab, r2_wf = WFCV(X, y, model, step_size=step_size, fold_size=fold_size)
    
    reg = sm.OLS(y_truth, sm.add_constant(y_pred)).fit()

    return {
        'SYMBOL': name_ticker,
        'MSE': float(np.mean(mse_tab)) if len(mse_tab) else np.nan,
        'OLS_R2': float(reg.rsquared),
        'OLS_Intercept': float(reg.params[0]),
        'OLS_Slope': float(reg.params[1]),
        'OLS_P_Value_Intercept': float(reg.pvalues[0]),
        'P_Value_Slope': float(reg.pvalues[1]),
        'n_samples': int(len(df_ticker)),
    }

### 1 month window

In [ ]:
if __name__ == "__main__":
    os.makedirs('Resultats', exist_ok=True)

    FOLD_SIZE = 21   
    STEP_SIZE = 5    
    MIN_REQUIS = FOLD_SIZE + STEP_SIZE + 10

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    df_all.drop(columns=[c for c in COLS_TO_DROP if c in df_all.columns], inplace=True, errors='ignore')
    
    all_tickers = df_all.columns[1:].tolist()

    valid_tickers = [col for col in all_tickers if df_all[col].count() >= MIN_REQUIS]
    print(f"Filtering completed: {len(valid_tickers)} tickers kept (those with >= {MIN_REQUIS} valid days).")

    rows = []
    csv_name = "Resultats/resultats_all_tickers_SVM_f1month.csv"

    for name in tqdm(valid_tickers, desc='Training SVM — 1 Month'):
        try:
            res = stats_forecasting_svm_custom(
                df_all=df_all, 
                name_ticker=name, 
                model_params=BEST_PARAMS_SVM,
                step_size=STEP_SIZE, 
                fold_size=FOLD_SIZE
            )
            rows.append(res)
        except ValueError as e:
            continue
        except Exception as e:
            print(f"\n[ERROR] {name}: {e}")
            continue
            
    df_final = pd.DataFrame(rows)
    df_final.to_csv(csv_name, index=False)
    print(f"\nAll SVM calculations for 1 MONTH completed successfully! Saved to: {csv_name}")

Filtering completed: 1035 tickers kept (those with >= 36 valid days).


Training SVM — 1 Month:   3%|▎         | 26/1035 [02:31<2:24:10,  8.57s/it]

### 1 year window

In [ ]:
if __name__ == "__main__":
    os.makedirs('Resultats', exist_ok=True)

    FOLD_SIZE = 200  
    STEP_SIZE = 50   
    MIN_REQUIS = FOLD_SIZE + STEP_SIZE + 20

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    df_all.drop(columns=[c for c in COLS_TO_DROP if c in df_all.columns], inplace=True, errors='ignore')
    
    all_tickers = df_all.columns[1:].tolist()

    valid_tickers = [col for col in all_tickers if df_all[col].count() >= MIN_REQUIS]
    print(f"Filtering completed: {len(valid_tickers)} tickers kept (those with >= {MIN_REQUIS} valid days).")

    rows = []
    csv_name = "Resultats/resultats_all_tickers_SVM_f1year.csv"

    for name in tqdm(valid_tickers, desc='Training SVM — 1 Year'):
        try:
            res = stats_forecasting_svm_custom(
                df_all=df_all, 
                name_ticker=name, 
                model_params=BEST_PARAMS_SVM,
                step_size=STEP_SIZE, 
                fold_size=FOLD_SIZE
            )
            rows.append(res)
        except ValueError as e:
            continue
        except Exception as e:
            print(f"\n[ERROR] {name}: {e}")
            continue
            
    df_final = pd.DataFrame(rows)
    df_final.to_csv(csv_name, index=False)
    print(f"\nAll SVM calculations for 1 YEAR completed successfully! Saved to: {csv_name}")

1035 tickers disponibles apres filtrage.
Lancement de l'analyse SVM sur 1035 tickers...


c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\diane\time-series-pred\.venv\Lib\site-packages\pa

Traitement terminé. Fichier enregistré sous : Resultats/resultats_all_tickers_SVM_opti.csv


,SYMBOL,MSE,OLS_R2,OLS_Intercept,OLS_Slope,OLS_P_Value_Intercept,P_Value_Slope,n_samples
0,ADBF Index,0.000312,0.022723,0.000461,0.271186,0.106137,5.963624e-16,3061
1,ADCM Index,0.000574,0.076022,0.000488,0.502061,0.231216,1.922420e-47,2861
2,ADCT Index,0.000339,0.046357,0.000315,0.444822,0.296588,3.048980e-31,3054
3,ADEG Index,0.000823,0.039739,0.000175,0.386455,0.710507,3.718765e-25,2888
4,ADHC Index,0.001092,0.029866,0.001221,0.275644,0.428031,2.299986e-04,692


### 5 years window

In [ ]:
if __name__ == "__main__":
    os.makedirs('Resultats', exist_ok=True)

    FOLD_SIZE = 1260 
    STEP_SIZE = 250  
    MIN_REQUIS = FOLD_SIZE + STEP_SIZE + 100 

    df_all = pd.read_csv('Datasets/returns_all.csv')
    df_all['Date'] = pd.to_datetime(df_all['Date'], errors='coerce')
    df_all.drop(columns=[c for c in COLS_TO_DROP if c in df_all.columns], inplace=True, errors='ignore')
    
    all_tickers = df_all.columns[1:].tolist()

    valid_tickers = [col for col in all_tickers if df_all[col].count() >= MIN_REQUIS]
    print(f"Filtering completed: {len(valid_tickers)} tickers kept (those with >= {MIN_REQUIS} valid days).")

    rows = []
    csv_name = "Resultats/resultats_all_tickers_SVM_f5years.csv"

    for name in tqdm(valid_tickers, desc='Training SVM — 5 Years'):
        try:
            res = stats_forecasting_svm_custom(
                df_all=df_all, 
                name_ticker=name, 
                model_params=BEST_PARAMS_SVM,
                step_size=STEP_SIZE, 
                fold_size=FOLD_SIZE
            )
            rows.append(res)
        except ValueError as e:
            continue
        except Exception as e:
            print(f"\n[ERROR] {name}: {e}")
            continue
            
    df_final = pd.DataFrame(rows)
    df_final.to_csv(csv_name, index=False)
    print(f"\nAll SVM calculations for 5 YEARS completed successfully! Saved to: {csv_name}")